In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
import os
import time

In [ ]:
CSV_FILE = 'advanced_hand_data.csv'
if os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE)
    print(f"[+] CSV file loaded. Current shape: {df.shape}")
    print(df['Target_Label'].value_counts())
else:
    # 42 coordinates (21 joints * 2 [x,y]) + 1 label column
    columns = [str(i) for i in range(42)] + ['Target_Label']
    df = pd.DataFrame(columns=columns)
    df.to_csv(CSV_FILE, index=False)
    print("[+] Created new hand landmark dataset CSV sheet.")

In [ ]:
# Setup MediaPipe Hand Landmarker
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7
)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

print("=== LANDMARK COLLECTION CONSOLE ===")
print("Press '0' -> Record FIST")
print("Press '1' -> Record PEACE SIGN")
print("Press '2' -> Record OPEN PALM")
print("Press 'q' -> Exit Loop")

recorded_counts = {0: 0, 1: 0, 2: 0}

with vision.HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        timestamp_ms = int(time.time() * 1000)
        result = landmarker.detect_for_video(mp_image, timestamp_ms)
        
        current_landmarks = None
        
        if result.hand_landmarks:
            current_landmarks = result.hand_landmarks[0]
            # Draw simple landmark points for visual feedback
            for lm in current_landmarks:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
        
        cv2.putText(frame, f"Fists: {recorded_counts[0]} | Peace: {recorded_counts[1]} | Palm: {recorded_counts[2]}", 
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.imshow("Landmark Collector Workspace", frame)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key in [ord('0'), ord('1'), ord('2')]:
            label = int(chr(key))
            if current_landmarks is not None:
                # wrist node relative translation-invariant translation
                wrist = current_landmarks[0]
                row_data = []
                for lm in current_landmarks:
                    row_data.extend([lm.x - wrist.x, lm.y - wrist.y])
                row_data.append(label)
                
                # Save to CSV instantly
                row_df = pd.DataFrame([row_data])
                row_df.to_csv(CSV_FILE, mode='a', header=False, index=False)
                
                recorded_counts[label] += 1
                print(f"[+] Recorded {label} landmark signature to {CSV_FILE}.")
            else:
                print("[-] Hand was not detected. Position hand in camera view.")

cap.release()
cv2.destroyAllWindows()
print("=== COLLECTION TASK COMPLETE ===")